# Milestone 1 lab, notebook 7 of 7: The production version

## Where we are

| Notebook | What you do |
|---|---|
| 1. The question and the source | Read the question. Meet the order system. Look at one order |
| 2. Read every order | Follow the pages of the API to the end |
| 3. The answer by hand | Work out the answer with plain Python. Meet the time zone trap |
| 4. Transform, then load | Put the orders in a warehouse table. Ask it one SQL question |
| 5. Every morning | Run it again the next day. Watch an old number change |
| 6. Why batch, and where it breaks | Measure how late the report is. Then watch the load fail |
| **7. The production version** (you are here) | The same logic as a tested program |

A notebook is a place to work things out. It is not a place to run a business from. Cells can run in the wrong order. Nothing checks the result.

So the same logic also exists as a small program, in the `daily_orders/` folder beside this lab. This notebook shows that it gives the same answers.

In [1]:
import inspect

from lab_helpers import LAB_DIR, api, load_result, set_morning, show_doc

from daily_orders import extract, pipeline, report, transform

## Step 1: The same parts, as files

In [2]:
show_doc("LLD.md", "1. Files")

**From `docs/LLD.md`**

> ## 1. Files
> 
> Everything is inside `src/m1_report_by_hand/`. Nothing imports from another milestone.
> 
> ```
> docs/                     BRD.md, HLD.md, LLD.md
> notebooks/                the lab: seven notebooks, run in order
>   lab_helpers.py          small helpers the notebooks share
>   results/answer_key.json the answer worked out by hand in notebook 3
> daily_orders/             production code
>   config.py               the API address, the time zone, the warehouse path
>   extract.py              read every page of orders
>   transform.py            orders → rows of the orders table (pandas)
>   load.py                 the warehouse tables, and the load
>   report.py               the report SQL and the manager's answer
>   pipeline.py             one morning, start to end
>   flow_data.py            builds the numbers shown on the flow page and the recap page
> tests/                    one test file per code file
> data/warehouse/           the DuckDB file (made when you run; not in Git)
> milestone_1_flow.html     the whole pipeline, animated
> milestone_1_recap.html    the closing page
> ```
> 
> The order system is shared by all milestones. It is in [`order_system/`](../../../order_system/) at the project root.

One file per job: extract, transform, load, report. Here is the extract function. Compare it with the loop you wrote in notebook 2.

In [3]:
print(inspect.getsource(extract.fetch_all_orders))

def fetch_all_orders(client: httpx.Client) -> Extract:
    """Follow the pages to the end.

    Each page holds up to 25 orders and a `next_cursor`.
    The last page has no `next_cursor`. That is the signal to stop.
    """
    orders: list[dict] = []
    cursor = None
    calls = 0
    while True:
        answer = client.get("/orders", params={"cursor": cursor} if cursor else None)
        calls += 1
        if answer.status_code == 429:
            raise RateLimited(
                f"The API refused call {calls} after {len(orders)} orders. "
                f"It said to wait {answer.headers.get('Retry-After', '?')} seconds. "
                "Nothing was loaded."
            )
        answer.raise_for_status()
        body = answer.json()
        orders += body["data"]
        cursor = body["next_cursor"]
        if cursor is None:
            return Extract(orders, calls)



It is your loop, with one thing added. When the API refuses a call (status 429), it stops with a clear error. It never loads half the orders.

The business day rule from notebook 3 is one small function too.

In [4]:
print(inspect.getsource(transform.business_day))

def business_day(placed_at: pd.Series) -> pd.Series:
    """The business day of each order: the date of `placed_at` on Arden's clock.

    `placed_at` is in UTC. 23:58 on Saturday in Arden is already Sunday in UTC,
    so the time is moved to Arden's time zone before the date is taken.
    """
    return placed_at.dt.tz_convert(BUSINESS_TZ).dt.date



## Step 2: Run it for Monday

`pipeline.run` does a whole morning: extract, transform, load, report.

In [5]:
LAB_PRODUCTION = LAB_DIR / "lab_production.duckdb"
LAB_PRODUCTION.unlink(missing_ok=True)

set_morning("2026-06-08")
monday = pipeline.run(api, LAB_PRODUCTION)

print(f"Loaded {monday.orders_loaded} orders in {monday.api_calls} API calls.\n")
print(report.format_answer(monday.answer))

It is now Monday 2026-06-08 07:00:00 in Arden.
Loaded 226 orders in 10 API calls.

Arden daily orders. Report for the morning of Mon 2026-06-08

  Yesterday   Sun 2026-06-07    30 orders      $807.11
  Day before  Sat 2026-06-06    33 orders      $786.71
  Change                         -3 orders      +$20.40

  Yesterday by city
    LKS     6 orders      $144.98
    NGT    10 orders      $355.26
    RVB    14 orders      $306.87

  Earlier days: there is no earlier report to compare with.


## Step 3: Hold it up against your answer key

The key is the file you made by hand in notebook 3.

In [6]:
key = load_result("answer_key")

checks = {
    "Yesterday, orders": (monday.answer["yesterday"]["orders"], key["yesterday"]["orders"]),
    "Yesterday, money": (
        monday.answer["yesterday"]["revenue_cents"],
        key["yesterday"]["revenue_cents"],
    ),
    "Day before, orders": (monday.answer["day_before"]["orders"], key["day_before"]["orders"]),
    "Day before, money": (
        monday.answer["day_before"]["revenue_cents"],
        key["day_before"]["revenue_cents"],
    ),
    "Change": (monday.answer["change"], key["change"]),
}
for city, row in key["yesterday"]["by_city"].items():
    program = monday.answer["yesterday"]["by_city"][city]
    checks[f"Yesterday, {city}"] = (
        (program["orders"], program["revenue_cents"]),
        (row["orders"], row["revenue_cents"]),
    )

for name, (program, by_hand) in checks.items():
    print(
        f"{'✅' if program == by_hand else '❌'} {name:<20} program: {program}   by hand: {by_hand}"
    )

✅ Yesterday, orders    program: 30   by hand: 30
✅ Yesterday, money     program: 80711   by hand: 80711
✅ Day before, orders   program: 33   by hand: 33
✅ Day before, money    program: 78671   by hand: 78671
✅ Change               program: {'orders': -3, 'revenue_cents': 2040}   by hand: {'orders': -3, 'revenue_cents': 2040}
✅ Yesterday, LKS       program: (6, 14498)   by hand: (6, 14498)
✅ Yesterday, RVB       program: (14, 30687)   by hand: (14, 30687)
✅ Yesterday, NGT       program: (10, 35526)   by hand: (10, 35526)


Every line matches. A test in `tests/test_notebook_parity.py` makes the same comparison every time the tests run.

## Step 4: Run it for Tuesday

Move the clock one day and run again. This time the fourth part of the question has an answer.

In [7]:
set_morning("2026-06-09")
tuesday = pipeline.run(api, LAB_PRODUCTION)

print(f"Loaded {tuesday.orders_loaded} orders in {tuesday.api_calls} API calls.\n")
print(report.format_answer(tuesday.answer))

It is now Tuesday 2026-06-09 07:00:00 in Arden.
Loaded 251 orders in 11 API calls.

Arden daily orders. Report for the morning of Tue 2026-06-09

  Yesterday   Mon 2026-06-08    25 orders      $591.75
  Day before  Sun 2026-06-07    30 orders      $807.11
  Change                         -5 orders     -$215.36

  Yesterday by city
    LKS     5 orders      $111.09
    NGT    10 orders      $251.23
    RVB    10 orders      $229.43

  Earlier days that changed since the report of Mon 2026-06-08:
    Fri 2026-06-05   +0 orders   -$5.78
      because order 1130: refund of $5.78


Read the last three lines. The program found what you found in notebook 5: Friday went down by $5.78, because order 1130 was refunded.

You can also run the program without a notebook. From this milestone's folder, in a terminal:

```
uv run python -m daily_orders
```

## Step 5: The tests

The program has tests. They start their own private order system, so they need nothing running.

In [8]:
!cd .. && uv run pytest -q --color=no 2>&1 | tail -n 2

FAILED tests/test_notebook_parity.py::test_the_notebooks_were_saved_with_their_outputs
1 failed, 33 passed in 3.10s


What the tests prove:

In [9]:
show_doc("LLD.md", "7. Checks")

**From `docs/LLD.md`**

> ## 7. Checks
> 
> | Check | Where |
> |---|---|
> | The report equals the answer key, which is worked out from the source by different code | `tests/test_pipeline.py` |
> | The notebook's hand-worked answer equals the production answer | `tests/test_notebook_parity.py` |
> | Order 1196 lands on Saturday | `tests/test_transform.py` |
> | Running the same morning twice gives the same tables | `tests/test_load.py` |
> | A 429 loads nothing and keeps the old table | `tests/test_pipeline.py` |
> | Every number on the flow page and the recap page comes from the code | `tests/test_flow_page.py` |

## Step 6: What is still wrong

In [10]:
show_doc("LLD.md", "8. What is still wrong")

**From `docs/LLD.md`**

> ## 8. What is still wrong
> 
> Measured with this code, one full load per morning:
> 
> | Morning | Orders read | API calls | Result |
> |---|---|---|---|
> | 06-08 | 226 | 10 | ✅ |
> | 06-09 | 251 | 11 | ✅ |
> | 06-10 | 282 | 12 | ✅ the last call allowed |
> | 06-11 | 317 | 13 | ❌ call 13 is refused: `429` |
> 
> On 06-09 only 25 orders were new. The load still read 251.
> 
> **It reloads every order every day. It is slow, and it hits the API's rate limit.** Milestone 2 loads only what changed.

The program does not hide this. On the morning of 11 June it stops and says why.

In [11]:
set_morning("2026-06-11")
try:
    pipeline.run(api, LAB_PRODUCTION)
except extract.RateLimited as error:
    print("The load failed:", error)

It is now Thursday 2026-06-11 07:00:00 in Arden.
The load failed: The API refused call 13 after 300 orders. It said to wait 10 seconds. Nothing was loaded.


## The words of milestone 1

| Word | In one or two sentences | You saw it in |
|---|---|---|
| **Data pipeline** | It moves data from where it is made to where it is used. It works in steps, and it runs on a schedule | Notebook 1 |
| **Source**, **destination** | Where the data is made, and where it is used | Notebook 1 |
| **Full load** | It reads every row on every run | Notebook 2 |
| **Event time** | When the thing happened in the real world. Business numbers use it | Notebook 3 |
| **Grain** | What one row of a table means | Notebook 3 |
| **ETL** | Transform first, then load only the result | Notebook 4 |
| **The five stages** | Ingest, store, transform, serve, observe | Notebook 4 |
| **Load time** | When the pipeline copied the row | Notebook 5 |
| **Latency** | The time from a change in the source until the warehouse can show it | Notebook 6 |
| **Batch** | One run handles a whole slice of data, then stops | Notebook 6 |
| **Freshness** | How new the data in the warehouse is | Notebook 6 |
| **Micro-batch**, **streaming** | The opposites of batch: a very small slice run again and again, and a job that never stops | Notebook 6 |

The project's [`GLOSSARY.md`](../../../GLOSSARY.md) keeps every word, with an interview line for each.

## What you now know

- The notebook logic and the program give the same answer, and a test checks that they always will.
- Each part of the program has one job, so each can be tested alone.
- The program fails loudly when the API refuses it. It never loads half a day.

**You have finished milestone 1.** Open [`milestone_1_recap.html`](../milestone_1_recap.html) in a browser for a look back at what you built. Then open [`milestone_1_flow.html`](../milestone_1_flow.html) again. You now know what every box on it does.

> **Still wrong:** it reloads every order every day. It is slow, and it hits the API's rate limit. Milestone 2 loads only what changed.